<a href="https://colab.research.google.com/github/6hamuge/Beyond-Sleep-Prediction/blob/main/PTDT_Sleep_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ─── 유틸리티 함수 ────────────────────────────────────────

def get_timestamp_col(df):
    """타임스탬프 컬럼 자동 탐지"""
    candidates = ['timestamp', 'time', 'datetime', 'ts', 'date']
    for c in df.columns:
        if any(k in c.lower() for k in candidates):
            return c
    # datetime dtype으로 탐지
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    return None


def extract_daily_features(df, sensor_name, ts_col=None, subject_col='subject_id'):
    """
    센서 데이터프레임 → 하루 단위 집계 피처 반환

    집계 통계: mean, std, min, max, count
    """
    df = df.copy()

    # 타임스탬프 처리
    if ts_col is None:
        ts_col = get_timestamp_col(df)

    if ts_col and ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], unit='ms', errors='coerce') \
                     if df[ts_col].dtype in ['int64','float64'] \
                     else pd.to_datetime(df[ts_col], errors='coerce')
        df['date'] = df[ts_col].dt.date
    elif 'date' not in df.columns:
        print(f'  ⚠️  [{sensor_name}] 타임스탬프 컬럼을 찾지 못했습니다.')
        return None

    df['date'] = pd.to_datetime(df['date'])

    # 수치형 컬럼만 선택 (subject_id, date 제외)
    exclude = {subject_col, 'date', ts_col}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c not in exclude]

    if not num_cols:
        print(f'  ⚠️  [{sensor_name}] 수치형 컬럼이 없습니다.')
        return None

    # 하루 단위 집계
    grp = df.groupby([subject_col, 'date'])[num_cols]
    aggs = {
        'mean' : grp.mean(),
        'std'  : grp.std().fillna(0),
        'min'  : grp.min(),
        'max'  : grp.max(),
        'count': grp.count(),
    }

    # 멀티-집계 컬럼 병합
    result_parts = []
    for agg_name, agg_df in aggs.items():
        agg_df.columns = [f'{sensor_name}__{c}__{agg_name}' for c in agg_df.columns]
        result_parts.append(agg_df)

    result = pd.concat(result_parts, axis=1).reset_index()
    print(f'  ✅ [{sensor_name}]: {result.shape[1]-2}개 피처, {result.shape[0]}행')
    return result


print('🔧 피처 엔지니어링 함수 정의 완료')

🔧 피처 엔지니어링 함수 정의 완료


In [ ]:
# ─── 모든 센서 → 일별 피처 추출 ──────────────────────────
print('📊 일별 피처 추출 시작...')

daily_features = {}
for key, df in raw.items():
    feat = extract_daily_features(df, sensor_name=key)
    if feat is not None:
        daily_features[key] = feat

print(f'\n✅ 피처 추출 완료: {len(daily_features)}개 센서')

📊 일별 피처 추출 시작...
  ✅ [ac_status]: 5개 피처, 700행
  ✅ [activity]: 5개 피처, 700행
  ⚠️  [ambience] 수치형 컬럼이 없습니다.
  ⚠️  [ble] 수치형 컬럼이 없습니다.
  ⚠️  [gps] 수치형 컬럼이 없습니다.
  ✅ [light]: 5개 피처, 700행
  ✅ [screen]: 5개 피처, 700행
  ⚠️  [usage_stats] 수치형 컬럼이 없습니다.
  ⚠️  [wifi] 수치형 컬럼이 없습니다.
  ⚠️  [hr] 수치형 컬럼이 없습니다.
  ✅ [w_light]: 5개 피처, 664행
  ✅ [pedo]: 35개 피처, 653행

✅ 피처 추출 완료: 6개 센서


In [ ]:
#데이터 모양 확인 코드
for key in ['ambience', 'ble', 'gps', 'usage_stats', 'wifi', 'hr']:
    print("\n====", key, "====")
    print(raw[key].dtypes)
    display(raw[key].head(5))
    print(raw[key].iloc[0].to_dict())


==== ambience ====
subject_id            object
timestamp     datetime64[ns]
m_ambience            object
dtype: object


,subject_id,timestamp,m_ambience
0,id01,2024-06-26 13:00:10,"[[Music, 0.30902618], [Vehicle, 0.081680894], ..."
1,id01,2024-06-26 13:02:10,"[[Music, 0.62307084], [Vehicle, 0.021118319], ..."
2,id01,2024-06-26 13:04:10,"[[Horse, 0.25209898], [Animal, 0.24263993], [C..."
3,id01,2024-06-26 13:06:10,"[[Speech, 0.93433166], [Inside, large room or ..."
4,id01,2024-06-26 13:08:10,"[[Speech, 0.8935082], [Inside, small room, 0.0..."


{'subject_id': 'id01', 'timestamp': Timestamp('2024-06-26 13:00:10'), 'm_ambience': array([array(['Music', '0.30902618'], dtype=object),
       array(['Vehicle', '0.081680894'], dtype=object),
       array(['Motor vehicle (road)', '0.04035286'], dtype=object),
       array(['Outside, urban or manmade', '0.037144363'], dtype=object),
       array(['Outside, rural or natural', '0.032663062'], dtype=object),
       array(['Car', '0.03199804'], dtype=object),
       array(['Speech', '0.029806137'], dtype=object),
       array(['Inside, large room or hall', '0.01684492'], dtype=object),
       array(['Truck', '0.016206821'], dtype=object),
       array(['Sound effect', '0.01591479'], dtype=object)], dtype=object)}

==== ble ====
subject_id            object
timestamp     datetime64[ns]
m_ble                 object
dtype: object


,subject_id,timestamp,m_ble
0,id01,2024-06-26 12:13:00,"[{'address': '00:15:7C:11:80:8D', 'device_clas..."
1,id01,2024-06-26 12:23:00,"[{'address': '0A:B1:26:4D:76:21', 'device_clas..."
2,id01,2024-06-26 12:33:00,"[{'address': '04:F5:AE:39:95:E0', 'device_clas..."
3,id01,2024-06-26 13:23:00,"[{'address': '06:C0:D2:6D:9F:69', 'device_clas..."
4,id01,2024-06-26 14:23:00,"[{'address': '10:2B:41:74:9F:B1', 'device_clas..."


{'subject_id': 'id01', 'timestamp': Timestamp('2024-06-26 12:13:00'), 'm_ble': array([{'address': '00:15:7C:11:80:8D', 'device_class': '0', 'rssi': -82},
       {'address': '01:B1:D2:20:9E:3A', 'device_class': '0', 'rssi': -61},
       {'address': '04:33:1F:D9:C1:50', 'device_class': '0', 'rssi': -86},
       {'address': '06:5C:2D:BC:39:BE', 'device_class': '0', 'rssi': -75},
       {'address': '09:42:21:0D:AD:DF', 'device_class': '0', 'rssi': -70},
       {'address': '0B:66:0D:D5:9C:4A', 'device_class': '0', 'rssi': -89},
       {'address': '10:B5:88:E7:85:69', 'device_class': '0', 'rssi': -89},
       {'address': '13:F0:CA:3B:DB:EF', 'device_class': '0', 'rssi': -77},
       {'address': '1A:23:C0:8F:43:4D', 'device_class': '0', 'rssi': -66},
       {'address': '24:11:53:BB:62:89', 'device_class': '1796', 'rssi': -37},
       {'address': '24:2D:F0:EE:1E:D0', 'device_class': '0', 'rssi': -85},
       {'address': '26:0C:48:28:15:77', 'device_class': '0', 'rssi': -63},
       {'address':

...

{'subject_id': 'id01', 'timestamp': Timestamp('2024-06-26 13:00:00'), 'm_usage_stats': array([{'app_name': '\xa0캐시워크', 'total_time': 69},
       {'app_name': 'NAVER', 'total_time': 549},
       {'app_name': '\xa0✝️성경일독Q', 'total_time': 7337}], dtype=object)}

==== wifi ====
subject_id            object
timestamp     datetime64[ns]
m_wifi                object
dtype: object


,subject_id,timestamp,m_wifi
0,id01,2024-06-26 12:03:00,"[{'bssid': 'a0:0f:37:9a:5d:8b', 'rssi': -78}, ..."
1,id01,2024-06-26 12:13:00,"[{'bssid': 'a0:0f:37:9a:5d:8b', 'rssi': -79}, ..."
2,id01,2024-06-26 12:23:00,"[{'bssid': '10:e3:c7:0a:74:d1', 'rssi': -78}, ..."
3,id01,2024-06-26 12:33:00,"[{'bssid': '10:e3:c7:09:7f:bc', 'rssi': -80}, ..."
4,id01,2024-06-26 12:43:00,"[{'bssid': '56:46:ae:59:b1:13', 'rssi': -44}, ..."


{'subject_id': 'id01', 'timestamp': Timestamp('2024-06-26 12:03:00'), 'm_wifi': array([{'bssid': 'a0:0f:37:9a:5d:8b', 'rssi': -78},
       {'bssid': 'a0:0f:37:9a:5d:8c', 'rssi': -78},
       {'bssid': 'a0:0f:37:9a:5d:8d', 'rssi': -78},
       {'bssid': 'a0:0f:37:9a:5d:8e', 'rssi': -78},
       {'bssid': 'a0:0f:37:9a:5d:8f', 'rssi': -78},
       {'bssid': 'a0:0f:37:96:56:ef', 'rssi': -58},
       {'bssid': '88:36:6c:86:75:84', 'rssi': -72},
       {'bssid': 'a0:0f:37:96:56:ee', 'rssi': -58},
       {'bssid': 'a0:0f:37:96:56:ed', 'rssi': -58},
       {'bssid': '86:25:19:b5:b2:a5', 'rssi': -61},
       {'bssid': 'a0:0f:37:96:56:ec', 'rssi': -58},
       {'bssid': '1e:39:29:8e:fb:e9', 'rssi': -71},
       {'bssid': '52:c2:e8:c7:9b:e4', 'rssi': -82},
       {'bssid': 'a0:0f:37:96:56:eb', 'rssi': -58},
       {'bssid': '12:e3:c7:09:20:34', 'rssi': -88},
       {'bssid': '58:86:94:4a:08:b8', 'rssi': -82},
       {'bssid': '90:9f:33:28:d0:2e', 'rssi': -78},
       {'bssid': '00:26:66:bc:4e:18'

,subject_id,timestamp,heart_rate
0,id01,2024-06-26 12:23:00,"[134, 134, 135, 133, 134, 135, 134, 135, 134, ..."
1,id01,2024-06-26 12:24:00,"[123, 122, 121, 120, 121, 121, 120, 118, 119, ..."
2,id01,2024-06-26 12:25:00,"[120, 119, 117, 116, 119, 121, 123, 123, 121, ..."
3,id01,2024-06-26 12:26:00,"[125, 124, 124, 124, 125, 124, 124, 123, 123, ..."
4,id01,2024-06-26 12:27:00,"[116, 116, 117, 118, 116, 116, 116, 117, 115, ..."


{'subject_id': 'id01', 'timestamp': Timestamp('2024-06-26 12:23:00'), 'heart_rate': array([134, 134, 135, 133, 134, 135, 134, 135, 134, 133, 133, 133, 132,
       132, 131, 131, 131, 132, 132, 134, 134, 134, 132, 130, 128, 126,
       126, 126, 127, 129, 130, 129, 130, 130, 127, 127, 126, 125, 123])}


In [ ]:
def get_timestamp_col(df):
    for c in df.columns:
        if any(k in c.lower() for k in ['timestamp','time','datetime','ts','date']):
            return c
    for c in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            return c
    return None


def extract_daily_features(df, sensor_name, subject_col='subject_id'):
    df = df.copy()
    ts_col = get_timestamp_col(df)

    if ts_col and ts_col in df.columns:
        df[ts_col] = (pd.to_datetime(df[ts_col], unit='ms', errors='coerce')
                      if df[ts_col].dtype in ['int64','float64']
                      else pd.to_datetime(df[ts_col], errors='coerce'))
        df['date'] = df[ts_col].dt.date
    elif 'date' not in df.columns:
        print(f'  ⚠️  [{sensor_name}] 타임스탬프 없음')
        return None

    df['date'] = pd.to_datetime(df['date'])

    exclude  = {subject_col, 'date', ts_col}
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]

    if not num_cols:
        return None

    grp = df.groupby([subject_col, 'date'])[num_cols]
    parts = []

    for agg_name, agg_df in {
        'mean': grp.mean(),
        'std': grp.std().fillna(0),
        'min': grp.min(),
        'max': grp.max(),
        'count': grp.count()
    }.items():
        agg_df.columns = [f'{sensor_name}__{c}__{agg_name}' for c in agg_df.columns]
        parts.append(agg_df)

    result = pd.concat(parts, axis=1).reset_index()
    print(f'  ✅ [{sensor_name}]: {result.shape[1]-2}개 피처')
    return result


# ─── object/list 센서용 유틸 ───────────────────────────────

def ensure_datetime(df, ts_col='timestamp'):
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
    df['date'] = pd.to_datetime(df[ts_col].dt.date)
    df['hour'] = df[ts_col].dt.hour
    return df


def to_list(x):
    if x is None:
        return []
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    return []


# ─── GPS 전용 피처 ────────────────────────────────────────

def extract_gps_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)
    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        latitudes, longitudes, altitudes, speeds = [], [], [], []

        # 추가: 마지막 이동 시각 / 야간 이동 비율 계산용
        moving_times = []
        night_speed_flags = []

        for _, row in g.iterrows():
            ts = row['timestamp']
            hour = row['hour']

            for item in to_list(row['m_gps']):
                if isinstance(item, dict):
                    if item.get('latitude') is not None:
                        latitudes.append(float(item.get('latitude')))

                    if item.get('longitude') is not None:
                        longitudes.append(float(item.get('longitude')))

                    if item.get('altitude') is not None:
                        altitudes.append(float(item.get('altitude')))

                    if item.get('speed') is not None:
                        spd = float(item.get('speed'))

                        # GPS 튐 제거
                        if 0 <= spd <= 30:
                            speeds.append(spd)

                            # 이동 여부 기준
                            is_moving = spd > 0.5

                            if is_moving:
                                moving_times.append(ts)

                            # 야간 이동 여부: 0~5시
                            if 0 <= hour <= 5:
                                night_speed_flags.append(is_moving)

        latitudes = np.array(latitudes)
        longitudes = np.array(longitudes)
        altitudes = np.array(altitudes)
        speeds = np.array(speeds)

        # 마지막 이동 시각
        if len(moving_times) > 0:
            last_move_time = max(moving_times)

            last_movement_hour = (
                last_move_time.hour +
                last_move_time.minute / 60
            )

            # 새벽은 24시 이후로 변환
            if last_movement_hour < 6:
                last_movement_hour += 24

        else:
            last_movement_hour = 0

        feat = {
            'subject_id': subj,
            'date': date,

            'gps__record_count': len(g),
            'gps__point_count': len(speeds),

            'gps__speed_mean': np.mean(speeds) if len(speeds) else 0,
            'gps__speed_std': np.std(speeds) if len(speeds) else 0,
            'gps__speed_max': np.max(speeds) if len(speeds) else 0,
            'gps__moving_ratio': np.mean(speeds > 0.5) if len(speeds) else 0,

            'gps__lat_std': np.std(latitudes) if len(latitudes) else 0,
            'gps__lon_std': np.std(longitudes) if len(longitudes) else 0,
            'gps__altitude_mean': np.mean(altitudes) if len(altitudes) else 0,
            'gps__altitude_std': np.std(altitudes) if len(altitudes) else 0,

            # 추가 피처
            'gps__last_movement_hour': last_movement_hour,
            'gps__night_moving_ratio': np.mean(night_speed_flags) if len(night_speed_flags) else 0,
        }

        if len(latitudes) and len(longitudes):
            rough_places = set(zip(np.round(latitudes, 3), np.round(longitudes, 3)))
            feat['gps__unique_rough_place_count'] = len(rough_places)
        else:
            feat['gps__unique_rough_place_count'] = 0

        rows.append(feat)

    result = pd.DataFrame(rows)
    print(f'  ✅ [gps_special]: {result.shape[1]-2}개 피처')
    return result


# ─── Ambience 전용 피처 ───────────────────────────────────

def extract_ambience_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)

    target_labels = [
        'Speech',
        'Music',
        'Vehicle',
        'Motor vehicle (road)',
        'Car',
        'Outside, urban or manmade',
        'Outside, rural or natural',
        'Inside, large room or hall',
        'Inside, small room',
        'Animal',
        'Silence',
        'Noise'
    ]

    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        sums = {label: 0.0 for label in target_labels}
        maxs = {label: 0.0 for label in target_labels}
        top1_counts = {label: 0 for label in target_labels}

        total_events = 0
        evening_events = 0
        evening_speech = 0.0
        evening_music = 0.0
        evening_vehicle = 0.0

        for _, row in g.iterrows():
            arr = to_list(row['m_ambience'])
            hour = row['hour']

            scores = {}

            for item in arr:
                item = to_list(item)
                if len(item) >= 2:
                    label = str(item[0])
                    try:
                        score = float(item[1])
                    except:
                        score = 0.0

                    scores[label] = score

                    if label in target_labels:
                        sums[label] += score
                        maxs[label] = max(maxs[label], score)

            if scores:
                total_events += 1
                top_label = max(scores, key=scores.get)

                if top_label in top1_counts:
                    top1_counts[top_label] += 1

                if 20 <= hour <= 23:
                    evening_events += 1
                    evening_speech += scores.get('Speech', 0.0)
                    evening_music += scores.get('Music', 0.0)
                    evening_vehicle += (
                        scores.get('Vehicle', 0.0)
                        + scores.get('Motor vehicle (road)', 0.0)
                    )

        feat = {
            'subject_id': subj,
            'date': date,
            'ambience__record_count': len(g),
            'ambience__valid_event_count': total_events,
            'ambience__evening_event_count': evening_events,
            'ambience__evening_speech_mean': evening_speech / evening_events if evening_events else 0,
            'ambience__evening_music_mean': evening_music / evening_events if evening_events else 0,
            'ambience__evening_vehicle_mean': evening_vehicle / evening_events if evening_events else 0,
        }

        for label in target_labels:
            clean = (
                label.lower()
                .replace(' ', '_')
                .replace(',', '')
                .replace('(', '')
                .replace(')', '')
                .replace('/', '_')
            )

            feat[f'ambience__{clean}__mean_score'] = sums[label] / total_events if total_events else 0
            feat[f'ambience__{clean}__max_score'] = maxs[label]
            feat[f'ambience__{clean}__top1_ratio'] = top1_counts[label] / total_events if total_events else 0

        rows.append(feat)

    result = pd.DataFrame(rows).fillna(0)
    print(f'  ✅ [ambience_special]: {result.shape[1]-2}개 피처')
    return result


# ─── BLE 전용 피처 ────────────────────────────────────────

def extract_ble_special_features(df, subject_col='subject_id'):
    df = ensure_datetime(df)
    rows = []

    for (subj, date), g in df.groupby([subject_col, 'date']):
        scan_device_counts = []
        rssis = []
        unique_addresses = set()
        device_classes = set()
        evening_counts = []
        night_counts = []

        for _, row in g.iterrows():
            devices = to_list(row['m_ble'])
            hour = row['hour']
            count = 0

            for dev in devices:
                if isinstance(dev, dict):
                    count += 1

                    addr = dev.get('address')
                    if addr:
                        unique_addresses.add(addr)

                    device_class = dev.get('device_class')
                    if device_class is not None:
                        device_classes.add(str(device_class))

                    rssi = dev.get('rssi')
                    if rssi is not None:
                        rssis.append(float(rssi))

            scan_device_counts.append(count)

            if 20 <= hour <= 23:
                evening_counts.append(count)

            if 0 <= hour <= 5:
                night_counts.append(count)

        scan_device_counts = np.array(scan_device_counts)
        rssis = np.array(rssis)

        feat = {
            'subject_id': subj,
            'date': date,
            'ble__scan_count': len(g),
            'ble__device_count_mean': np.mean(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__device_count_std': np.std(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__device_count_max': np.max(scan_device_counts) if len(scan_device_counts) else 0,
            'ble__unique_device_count': len(unique_addresses),
            'ble__unique_device_class_count': len(device_classes),
            'ble__rssi_mean': np.mean(rssis) if len(rssis) else 0,
            'ble__rssi_std': np.std(rssis) if len(rssis) else 0,
            'ble__rssi_max': np.max(rssis) if len(rssis) else 0,
            'ble__strong_signal_ratio': np.mean(rssis > -60) if len(rssis) else 0,
            'ble__evening_device_count_mean': np.mean(evening_counts) if len(evening_counts) else 0,
            'ble__night_device_count_mean': np.mean(night_counts) if len(night_counts) else 0,
        }

        rows.append(feat)

    result = pd.DataFrame(rows)
    print(f'  ✅ [ble_special]: {result.shape[1]-2}개 피처')
    return result


# ─── 일별 피처 추출 ───────────────────────────────────────

print('📊 일별 피처 추출...')

daily_features = {k: extract_daily_features(df, k) for k, df in raw.items()}
daily_features = {k: v for k, v in daily_features.items() if v is not None}

# object/list 센서 전용 피처 추가
daily_features['gps_special'] = extract_gps_special_features(raw['gps'])
daily_features['ambience_special'] = extract_ambience_special_features(raw['ambience'])
daily_features['ble_special'] = extract_ble_special_features(raw['ble'])

from functools import reduce

feature_df = reduce(
    lambda l, r: pd.merge(l, r, on=['subject_id', 'date'], how='outer'),
    daily_features.values()
)

print(f'\n✅ 통합 피처: {feature_df.shape}')

📊 일별 피처 추출...
  ✅ [ac_status]: 5개 피처
  ✅ [activity]: 5개 피처
  ✅ [light]: 5개 피처
  ✅ [screen]: 5개 피처
  ✅ [w_light]: 5개 피처
  ✅ [pedo]: 35개 피처
  ✅ [gps_special]: 13개 피처
  ✅ [ambience_special]: 42개 피처
  ✅ [ble_special]: 12개 피처

✅ 통합 피처: (700, 129)


In [ ]:
gps_cols = [c for c in feature_df.columns if c.startswith('gps__')]

display(
    feature_df[
        ['subject_id','date'] + gps_cols
    ].head()
)

,subject_id,date,gps__record_count,gps__point_count,gps__speed_mean,gps__speed_std,gps__speed_max,gps__moving_ratio,gps__lat_std,gps__lon_std,gps__altitude_mean,gps__altitude_std,gps__last_movement_hour,gps__night_moving_ratio,gps__unique_rough_place_count
0,id01,2024-06-26,707.0,6017.0,0.660462,2.334951,28.2200,0.162872,0.010646,0.023181,91.971032,13.699433,23.983333,0.000000,131.0
1,id01,2024-06-27,1439.0,11867.0,1.051500,3.168291,29.6777,0.213281,0.007768,0.022045,94.204758,15.050607,23.983333,0.121286,196.0
2,id01,2024-06-28,1418.0,11618.0,0.798394,2.783343,29.6666,0.154502,0.008652,0.021288,92.263865,12.256198,23.983333,0.045687,221.0
3,id01,2024-06-29,1440.0,11542.0,0.561169,2.046405,28.0971,0.172587,0.004166,0.008380,101.654223,10.319966,22.833333,0.153486,125.0
4,id01,2024-06-30,1440.0,12604.0,0.415772,1.981309,27.7601,0.096953,0.002223,0.016533,110.763393,13.936473,23.450000,0.002514,131.0


In [ ]:
amb_cols = [c for c in feature_df.columns if c.startswith('ambience__')]

display(
    feature_df[
        ['subject_id','date'] + amb_cols
    ].head()
)

,subject_id,date,ambience__record_count,ambience__valid_event_count,ambience__evening_event_count,ambience__evening_speech_mean,ambience__evening_music_mean,ambience__evening_vehicle_mean,ambience__speech__mean_score,ambience__speech__max_score,...,ambience__inside_small_room__top1_ratio,ambience__animal__mean_score,ambience__animal__max_score,ambience__animal__top1_ratio,ambience__silence__mean_score,ambience__silence__max_score,ambience__silence__top1_ratio,ambience__noise__mean_score,ambience__noise__max_score,ambience__noise__top1_ratio
0,id01,2024-06-26,304,304,94,6.655621e-02,3.375035e-02,0.065919,2.466539e-01,9.776402e-01,...,0.042763,0.019676,0.467283,0.026316,0.116573,1.0,0.115132,0.004826,0.470528,0.0
1,id01,2024-06-27,720,720,120,8.634859e-08,7.066516e-15,0.000000,8.622866e-08,8.634999e-08,...,0.000000,0.000000,0.000000,0.000000,0.998611,1.0,0.998611,0.000000,0.000000,0.0
2,id01,2024-06-28,710,710,120,8.307381e-03,7.007628e-15,0.000000,1.434175e-03,9.968755e-01,...,0.000000,0.000146,0.103939,0.000000,0.995774,1.0,0.995775,0.000000,0.000000,0.0
3,id01,2024-06-29,720,720,120,8.634858e-08,7.066518e-15,0.000000,8.622867e-08,8.634974e-08,...,0.000000,0.000379,0.273025,0.000000,0.998611,1.0,0.998611,0.000000,0.000000,0.0
4,id01,2024-06-30,720,720,120,8.562902e-08,7.007629e-15,0.000366,8.622866e-08,8.635002e-08,...,0.000000,0.000087,0.062971,0.000000,0.998685,1.0,0.998611,0.000000,0.000000,0.0


In [ ]:
ble_cols = [c for c in feature_df.columns if c.startswith('ble__')]

display(
    feature_df[
        ['subject_id','date'] + ble_cols
    ].head()
)

,subject_id,date,ble__scan_count,ble__device_count_mean,ble__device_count_std,ble__device_count_max,ble__unique_device_count,ble__unique_device_class_count,ble__rssi_mean,ble__rssi_std,ble__rssi_max,ble__strong_signal_ratio,ble__evening_device_count_mean,ble__night_device_count_mean
0,id01,2024-06-26,34.0,30.823529,29.115864,116.0,949.0,6.0,-75.602099,9.647166,-27.0,0.065840,31.785714,0.0
1,id01,2024-06-27,55.0,19.854545,18.231260,88.0,951.0,4.0,-73.814103,10.365185,-34.0,0.118132,6.285714,7.0
2,id01,2024-06-28,57.0,21.175439,16.064697,46.0,1031.0,4.0,-75.840099,9.468832,-39.0,0.082850,4.111111,4.0
3,id01,2024-06-29,42.0,5.000000,5.814596,34.0,149.0,3.0,-72.328571,13.193131,-33.0,0.223810,4.333333,4.5
4,id01,2024-06-30,38.0,14.736842,20.645739,70.0,499.0,3.0,-75.530357,10.037206,-35.0,0.073214,4.875000,1.0


In [ ]:
feature_df[gps_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
gps__record_count,660.0,1213.046970,311.127373,1.000000,1039.250000,1387.000000,1438.000000,1440.000000
gps__point_count,660.0,12916.957576,4164.357039,2.000000,10146.750000,13104.000000,15815.000000,22714.000000
gps__speed_mean,660.0,0.739777,0.766879,0.018713,0.285401,0.543199,0.963256,6.185291
gps__speed_std,660.0,2.159134,1.558983,0.061946,1.186427,1.851572,2.756334,10.663510
gps__speed_max,660.0,22.265047,7.559319,0.235800,17.803925,24.543250,29.085100,30.000000
gps__moving_ratio,660.0,0.164415,0.135843,0.000000,0.076452,0.126571,0.209015,1.000000
gps__lat_std,660.0,0.049341,0.123327,0.000002,0.004886,0.009891,0.019420,0.774912
gps__lon_std,660.0,0.037664,0.093225,0.000004,0.005929,0.014901,0.035659,0.971039
gps__altitude_mean,660.0,100.385117,21.495829,8.387962,89.869630,98.982867,111.237157,262.403367
gps__altitude_std,660.0,18.778164,21.983242,0.000000,7.930922,11.779862,22.052472,219.628086


In [ ]:
feature_df[amb_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
ambience__record_count,700.0,680.824286,88.873505,1.790000e+02,6.910000e+02,7.200000e+02,720.000000,720.000000
ambience__valid_event_count,700.0,680.824286,88.873505,1.790000e+02,6.910000e+02,7.200000e+02,720.000000,720.000000
ambience__evening_event_count,700.0,112.230000,23.140300,0.000000e+00,1.200000e+02,1.200000e+02,120.000000,120.000000
ambience__evening_speech_mean,700.0,0.081833,0.136822,0.000000e+00,8.634859e-08,8.634879e-08,0.165618,0.627408
ambience__evening_music_mean,700.0,0.017279,0.038975,0.000000e+00,7.066453e-15,7.066517e-15,0.017654,0.334563
ambience__evening_vehicle_mean,700.0,0.011941,0.029780,0.000000e+00,0.000000e+00,0.000000e+00,0.007975,0.226780
ambience__speech__mean_score,700.0,0.066319,0.099417,7.133181e-08,8.634859e-08,1.446734e-04,0.143833,0.477618
ambience__speech__max_score,700.0,0.439152,0.467636,8.634858e-08,8.635003e-08,1.021244e-01,0.986782,0.999633
ambience__speech__top1_ratio,700.0,0.108814,0.159616,0.000000e+00,0.000000e+00,1.388889e-03,0.247222,0.727273
ambience__music__mean_score,700.0,0.012419,0.024486,5.827283e-15,7.056702e-15,7.066517e-15,0.018138,0.301799


In [ ]:
feature_df[ble_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
ble__scan_count,651.0,33.533026,17.107265,1.000000,19.000000,32.000000,47.000000,98.000000
ble__device_count_mean,651.0,19.304116,15.140716,2.000000,7.613300,13.406250,27.514286,73.470588
ble__device_count_std,651.0,14.943434,11.800497,0.000000,3.694177,13.402901,24.185502,60.451893
ble__device_count_max,651.0,55.371736,37.685705,2.000000,18.000000,53.000000,87.000000,196.000000
ble__unique_device_count,651.0,562.139785,643.985636,2.000000,132.500000,315.000000,726.000000,3125.000000
ble__unique_device_class_count,651.0,3.007680,0.800924,2.000000,2.000000,3.000000,3.000000,7.000000
ble__rssi_mean,651.0,-76.906095,3.949763,-86.085859,-79.821356,-77.059222,-74.457453,-63.110294
ble__rssi_std,651.0,13.042987,2.988751,2.000000,10.490609,12.940007,14.831632,23.279432
ble__rssi_max,651.0,-31.993856,10.911108,-80.000000,-40.000000,-31.000000,-24.000000,-1.000000
ble__strong_signal_ratio,651.0,0.105228,0.069350,0.000000,0.052632,0.095238,0.144193,0.455882
